# TinyYOLO Experiment Suite — Notebook 4/7
## Ablation Studies (Table IX)

**What:** 7 ablation experiments validating each architectural choice.

**GPU Time:** ~18h T4 (7 runs × 100 epochs × ~2.5h each)

| # | Ablation | What changes |
|---|----------|-------------|
| 1 | Ghost vs Conv | Replace GhostConv with standard Conv |
| 2 | No attention | Remove all attention blocks |
| 3 | FPN only (no PAN) | Remove bottom-up path |
| 4 | No neck | Skip neck entirely |
| 5 | Resolution 320 | Smaller input |
| 6 | Resolution 640 | Larger input |
| 7 | No mosaic | Disable mosaic augmentation |

In [ ]:
import os, sys, socket, shutil
from pathlib import Path
REPO = 'https://github.com/ShMazumder/tinyYOLO.git'
try: socket.create_connection(('github.com', 80), timeout=5)
except: print('❌ No internet!'); sys.exit(1)
if Path('tinyYOLO').exists(): shutil.rmtree('tinyYOLO')
!git clone {REPO} --depth 1
%cd tinyYOLO
!pip install -e . -q 2>&1 | tail -1
!pip install tqdm timm psutil -q 2>&1 | tail -1
import torch
if torch.cuda.is_available(): print(f'🔥 GPU: {torch.cuda.get_device_name(0)}')
!python -c "from ultralytics.data.utils import check_det_dataset; check_det_dataset('VOC.yaml')" 2>&1 | tail -3
print('✅ Setup complete!')

In [ ]:
# Baseline: quantized variant, 416, 100 epochs
!python scripts/train.py --task det --variant quantized --data voc.yaml --lr 2e-4 --no-amp --freeze-epochs 5 --grad-clip 5.0 --val-conf 0.001 --ema-decay 0.9998 --close-mosaic 40 \
    --imgsz 416 --epochs 100 --batch 32 --lr 2e-4 --no-amp --freeze-epochs 5 --grad-clip 5.0 --seed 42 --warmup 3 \
    --pretrained --name ablation-baseline

In [ ]:
# Ablation: No attention
!python scripts/train.py --task det --variant quantized --attention none --data voc.yaml --lr 2e-4 --no-amp --freeze-epochs 5 --grad-clip 5.0 --val-conf 0.001 --ema-decay 0.9998 --close-mosaic 40 \
    --imgsz 416 --epochs 100 --batch 32 --lr 2e-4 --no-amp --freeze-epochs 5 --grad-clip 5.0 --seed 42 --warmup 3 \
    --pretrained --name ablation-no-attention

# Ablation: SE attention instead of ECA
!python scripts/train.py --task det --variant quantized --attention se --data voc.yaml --lr 2e-4 --no-amp --freeze-epochs 5 --grad-clip 5.0 --val-conf 0.001 --ema-decay 0.9998 --close-mosaic 40 \
    --imgsz 416 --epochs 100 --batch 32 --lr 2e-4 --no-amp --freeze-epochs 5 --grad-clip 5.0 --seed 42 --warmup 3 \
    --pretrained --name ablation-se-attention

In [ ]:
# Ablation: Resolution 320
!python scripts/train.py --task det --variant quantized --data voc.yaml --lr 2e-4 --no-amp --freeze-epochs 5 --grad-clip 5.0 --val-conf 0.001 --ema-decay 0.9998 --close-mosaic 40 \
    --imgsz 320 --epochs 100 --batch 32 --lr 2e-4 --no-amp --freeze-epochs 5 --grad-clip 5.0 --seed 42 --warmup 3 \
    --pretrained --name ablation-res320

# Ablation: Resolution 640
!python scripts/train.py --task det --variant quantized --data voc.yaml --lr 2e-4 --no-amp --freeze-epochs 5 --grad-clip 5.0 --val-conf 0.001 --ema-decay 0.9998 --close-mosaic 40 \
    --imgsz 640 --epochs 100 --batch 16 --lr 2e-4 --no-amp --freeze-epochs 5 --grad-clip 5.0 --seed 42 --warmup 3 \
    --pretrained --name ablation-res640

In [ ]:
# Ablation: No mosaic augmentation (--quick disables mosaic)
!python scripts/train.py --task det --variant quantized --data voc.yaml --lr 2e-4 --no-amp --freeze-epochs 5 --grad-clip 5.0 --val-conf 0.001 --ema-decay 0.9998 --close-mosaic 40 \
    --imgsz 416 --epochs 100 --batch 32 --lr 2e-4 --no-amp --freeze-epochs 5 --grad-clip 5.0 --seed 42 --warmup 3 \
    --pretrained --quick --name ablation-no-mosaic

In [ ]:
# Ablation: SiLU activation (standard variant) vs ReLU6 baseline
!python scripts/train.py --task det --variant standard --data voc.yaml --lr 2e-4 --no-amp --freeze-epochs 5 --grad-clip 5.0 --val-conf 0.001 --ema-decay 0.9998 --close-mosaic 40 \
    --imgsz 416 --epochs 100 --batch 32 --lr 2e-4 --no-amp --freeze-epochs 5 --grad-clip 5.0 --seed 42 --warmup 3 \
    --pretrained --name ablation-silu

In [ ]:
import json, glob
from pathlib import Path

print('='*70)
print('ABLATION RESULTS (100 epochs, VOC, 416×416 unless noted)')
print('='*70)
print(f'{"Experiment":<30} {"mAP@50":>10}')
print('-'*70)
for name in ['ablation-baseline', 'ablation-no-attention', 'ablation-se-attention',
             'ablation-res320', 'ablation-res640', 'ablation-no-mosaic', 'ablation-silu']:
    s = Path(f'experiments/results/{name}/summary.json')
    if s.exists():
        with open(s) as f: data = json.load(f)
        m = data.get('best_map50', 0) * 100
        print(f'  {name:<30} {m:>8.1f}%')
    else:
        print(f'  {name:<30}      N/A')
print('='*70)